# Dashboard de Desempenho Comercial – Vendas
## Visualização de Dados e Design de Dashboards – Ciência de Dados para Negócios (FATEC VOTORANTIM)
- **Objetivo:** construir um dashboard interativo no Databricks que transforme dados comerciais em informações úteis para tomada de decisão, utilizando SQL, KPIs, visualizações adequadas, filtros/interações e organização visual.
- **Base de dados:** tabela `workspace.default.vendas` (planilha *Aula 2 Visualizacao de dados.xlsx*), com Data, Produto, Categoria, Cidade, Estado, Vendedor, Quantidade, Preço, Receita, Custo, Lucro e Canal de Venda.
- **Entrega:** Dashboard publicado no Databricks com o nome "Dashboard de Desempenho Comercial – Vendas", criado a partir deste notebook.
- **Fluxo:** DADOS → INFORMAÇÃO → VISUALIZAÇÃO → INSIGHT → DECISÃO

# Aluno: Bruno Felipe de Souza Araujo

## Parte 1 – Entendendo o problema

### Perguntas de negócio
1. A receita e o lucro estão crescendo de um ano para o outro, e em quais meses houve o melhor e o pior resultado?
2. Quais categorias de produto mais geram receita e quais têm a melhor margem de lucro?
3. Quanto da receita depende de cada estado e existe algum estado com margem diferente dos demais?
4. Quais canais de venda concentram o faturamento e algum canal tem pouca participação?
5. Quais vendedores trazem mais lucro para a empresa e o resultado da equipe é equilibrado?

### Objetivo principal do dashboard
Mostrar, em uma única tela, o desempenho comercial da empresa (receita, lucro e margem), a evolução ao longo do tempo e onde estão os melhores e piores resultados por categoria, estado, canal e vendedor, permitindo ao gestor decidir onde agir.

### Público-alvo
Diretoria e gerência comercial: pessoas que precisam de uma visão geral rápida, sem consultar a base original, e que querem poder filtrar por período, estado, categoria e canal.

## Preparação dos dados
A tabela `workspace.default.vendas` foi criada nas aulas anteriores a partir do upload da planilha (*Catalog > Create table*). Todas as consultas e o dashboard usam essa tabela diretamente, com os nomes de colunas originais.

> Se a tabela `vendas` ainda não existir: em *Catalog > workspace > default > Create > Table*, faça upload da aba **Base de Vendas** com o nome `vendas`.

In [0]:
%sql
-- Conferência da base: quantidade de registros e período disponível
SELECT
    COUNT(*) AS Total_Registros,
    MIN(Data) AS Data_Inicial,
    MAX(Data) AS Data_Final
FROM workspace.default.vendas

Total_Registros,Data_Inicial,Data_Final
10000,2025-01-01,2026-07-31


## Parte 2 – Definição dos KPIs

| KPI | Por que é importante? | Como será calculado? |
|---|---|---|
| **Receita total** | Mostra o tamanho do negócio e se a empresa está vendendo mais ou menos. É o primeiro número que o gestor olha. | `SUM(Receita)` |
| **Lucro total** | Receita sozinha não paga as contas: o lucro mostra quanto realmente sobra para a empresa depois do custo. | `SUM(Lucro)` |
| **Margem de lucro (%)** | Permite comparar a rentabilidade entre categorias, estados e canais de tamanhos diferentes; mostra se a receita alta vem acompanhada de lucro. | `SUM(Lucro) / SUM(Receita) * 100` |
| **Ticket médio** | Indica o valor médio de cada venda; ajuda a entender se o crescimento vem de vender mais vezes ou de vender itens mais caros. | `AVG(Receita)` (receita / nº de vendas) |

In [0]:
%sql
-- KPIs principais
SELECT
    ROUND(SUM(Receita), 2) AS Receita_Total,
    ROUND(SUM(Lucro), 2) AS Lucro_Total,
    ROUND(SUM(Lucro) / SUM(Receita) * 100, 2) AS Margem_Lucro_Percentual,
    ROUND(AVG(Receita), 2) AS Ticket_Medio,
    COUNT(*) AS Numero_Vendas
FROM workspace.default.vendas

Receita_Total,Lucro_Total,Margem_Lucro_Percentual,Ticket_Medio,Numero_Vendas
38032640.23,11055552.42,29.07,3803.26,10000


## Parte 3 – Consultas SQL
Cada consulta responde a uma das perguntas da Parte 1. Os resultados viram as visualizações do dashboard.

In [0]:
%sql
-- Pergunta 1: evolução mensal da receita, do lucro e da margem (análise temporal)
SELECT
    DATE_TRUNC('month', Data) AS Mes,
    ROUND(SUM(Receita), 2) AS Receita_Total,
    ROUND(SUM(Lucro), 2) AS Lucro_Total,
    ROUND(SUM(Lucro) / SUM(Receita) * 100, 2) AS Margem_Lucro_Percentual
FROM workspace.default.vendas
GROUP BY DATE_TRUNC('month', Data)
ORDER BY Mes

Mes,Receita_Total,Lucro_Total,Margem_Lucro_Percentual
2025-01-01T00:00:00.000Z,1749729.90,521291.42,29.79
2025-02-01T00:00:00.000Z,1862340.08,540883.44,29.04
2025-03-01T00:00:00.000Z,1685475.98,489641.15,29.05
2025-04-01T00:00:00.000Z,1990704.17,587376.87,29.51
2025-05-01T00:00:00.000Z,2112465.04,607346.91,28.75
2025-06-01T00:00:00.000Z,2119185.31,617373.07,29.13
2025-07-01T00:00:00.000Z,1901045.67,544371.20,28.64
2025-08-01T00:00:00.000Z,2303169.09,661459.17,28.72
2025-09-01T00:00:00.000Z,1954404.58,572631.40,29.30
2025-10-01T00:00:00.000Z,1891849.15,568409.49,30.05


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Pergunta 1 (complemento): comparação do mesmo período (jan a jul) entre 2025 e 2026
SELECT
    YEAR(Data) AS Ano,
    ROUND(SUM(Receita), 2) AS Receita_Total,
    ROUND(SUM(Lucro), 2) AS Lucro_Total,
    ROUND(SUM(Lucro) / SUM(Receita) * 100, 2) AS Margem_Lucro_Percentual
FROM workspace.default.vendas
WHERE MONTH(Data) <= 7
GROUP BY YEAR(Data)
ORDER BY Ano

Ano,Receita_Total,Lucro_Total,Margem_Lucro_Percentual
2025,13420946.15,3908284.06,29.12
2026,14509840.85,4216383.17,29.06


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Pergunta 2: receita, lucro e margem por categoria
SELECT
    Categoria,
    ROUND(SUM(Receita), 2) AS Receita_Total,
    ROUND(SUM(Lucro), 2) AS Lucro_Total,
    ROUND(SUM(Lucro) / SUM(Receita) * 100, 2) AS Margem_Lucro_Percentual,
    ROUND(SUM(Receita) * 100 / SUM(SUM(Receita)) OVER (), 2) AS Participacao_Receita_Percentual
FROM workspace.default.vendas
GROUP BY Categoria
ORDER BY Receita_Total DESC

Categoria,Receita_Total,Lucro_Total,Margem_Lucro_Percentual,Participacao_Receita_Percentual
Informática,17471236.76,4877004.83,27.91,45.94
Mobile,8308714.26,2253247.05,27.12,21.85
Componentes,6047641.98,1454307.29,24.05,15.90
Escritório,4654209.37,1725758.32,37.08,12.24
Acessórios,1550837.86,745234.93,48.05,4.08


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Pergunta 3: receita, lucro, margem e participação por estado
SELECT
    Estado,
    ROUND(SUM(Receita), 2) AS Receita_Total,
    ROUND(SUM(Lucro), 2) AS Lucro_Total,
    ROUND(SUM(Lucro) / SUM(Receita) * 100, 2) AS Margem_Lucro_Percentual,
    ROUND(SUM(Receita) * 100 / SUM(SUM(Receita)) OVER (), 2) AS Participacao_Receita_Percentual
FROM workspace.default.vendas
GROUP BY Estado
ORDER BY Receita_Total DESC

Estado,Receita_Total,Lucro_Total,Margem_Lucro_Percentual,Participacao_Receita_Percentual
SP,18823631.34,5477239.59,29.10,49.49
PR,5059251.82,1460601.49,28.87,13.30
RJ,4996189.47,1440785.89,28.84,13.14
SC,4677212.04,1371376.52,29.32,12.30
MG,4476355.56,1305548.93,29.17,11.77


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Pergunta 4: participação de cada canal de venda
SELECT
    `Canal de Venda`,
    COUNT(*) AS Numero_Vendas,
    ROUND(SUM(Receita), 2) AS Receita_Total,
    ROUND(SUM(Receita) * 100 / SUM(SUM(Receita)) OVER (), 2) AS Participacao_Receita_Percentual,
    ROUND(SUM(Lucro) / SUM(Receita) * 100, 2) AS Margem_Lucro_Percentual
FROM workspace.default.vendas
GROUP BY `Canal de Venda`
ORDER BY Receita_Total DESC

Canal de Venda,Numero_Vendas,Receita_Total,Participacao_Receita_Percentual,Margem_Lucro_Percentual
Site,3077,11333193.28,29.80,29.34
Loja Física,2778,10426125.47,27.41,29.10
Aplicativo,2196,8764829.56,23.05,28.90
Marketplace,1365,5496811.79,14.45,28.92
Televendas,584,2011680.13,5.29,28.53


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Pergunta 5: ranking de vendedores por lucro
SELECT
    Vendedor,
    COUNT(*) AS Numero_Vendas,
    ROUND(SUM(Receita), 2) AS Receita_Total,
    ROUND(SUM(Lucro), 2) AS Lucro_Total,
    ROUND(SUM(Lucro) / SUM(Receita) * 100, 2) AS Margem_Lucro_Percentual,
    ROUND(AVG(Receita), 2) AS Ticket_Medio
FROM workspace.default.vendas
GROUP BY Vendedor
ORDER BY Lucro_Total DESC

Vendedor,Numero_Vendas,Receita_Total,Lucro_Total,Margem_Lucro_Percentual,Ticket_Medio
Bruno Costa,849,3416125.54,989919.01,28.98,4023.70
Larissa Gomes,871,3364753.79,983095.47,29.22,3863.09
Eduarda Alves,844,3262199.53,965178.18,29.59,3865.17
Isabela Santos,860,3283272.78,950294.93,28.94,3817.76
Gabriela Souza,865,3194013.26,929418.10,29.10,3692.50
Diego Rocha,850,3171665.14,916148.30,28.89,3731.37
João Ribeiro,819,3142078.76,915829.60,29.15,3836.48
Felipe Martins,824,3138584.21,909014.53,28.96,3808.96
Carla Mendes,823,3151423.86,904179.00,28.69,3829.19
Henrique Silva,788,3032301.37,893749.37,29.47,3848.10


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Parte 5 (desafio dos filtros): o padrão das categorias se mantém em SP vendendo pelo Site?
SELECT
    Categoria,
    ROUND(SUM(Receita), 2) AS Receita_Total,
    ROUND(SUM(Lucro) / SUM(Receita) * 100, 2) AS Margem_Lucro_Percentual,
    ROUND(SUM(Receita) * 100 / SUM(SUM(Receita)) OVER (), 2) AS Participacao_Receita_Percentual
FROM workspace.default.vendas
WHERE Estado = 'SP'
  AND `Canal de Venda` = 'Site'
GROUP BY Categoria
ORDER BY Receita_Total DESC

Categoria,Receita_Total,Margem_Lucro_Percentual,Participacao_Receita_Percentual
Informática,2797451.37,27.70,47.82
Mobile,1229169.71,27.18,21.01
Escritório,810703.58,36.46,13.86
Componentes,773542.86,24.72,13.22
Acessórios,239372.94,48.18,4.09


Databricks visualization. Run in Databricks to view.

## Partes 4, 6 e 7 – Visualizações, layout e títulos

| # | Título no dashboard | Pergunta que responde | Tipo de gráfico | Por que esse tipo? |
|---|---|---|---|---|
| 1 | Evolução mensal da receita e do lucro | A receita e o lucro estão crescendo? Qual o melhor/pior mês? | Linha | Linha é o melhor gráfico para mostrar tendência ao longo do tempo. |
| 2 | Receita por categoria de produto | Quais categorias mais vendem? | Barras horizontais ordenadas | Barras facilitam comparar categorias; ordenar mostra o ranking. |
| 3 | Margem de lucro por categoria | Receita alta significa lucro alto? | Barras ordenadas | Comparação direta da rentabilidade entre categorias. |
| 4 | Receita por estado | Quais regiões concentram as vendas e quais merecem atenção? | Barras ordenadas | Comparação direta entre poucos estados (5). |
| 5 | Receita por canal de venda | Existe concentração em algum canal? | Barras ordenadas | Comparação entre categorias; mais fácil de ler que pizza com 5 fatias. |
| 6 | Ranking de vendedores por lucro | Quais vendedores se destacam? | Barras horizontais (ranking) | Nomes longos ficam legíveis na horizontal e a ordem mostra o ranking. |

**Layout (hierarquia visual):**
```
TÍTULO DO DASHBOARD
FILTROS: Período | Estado | Categoria | Canal de venda
KPI Receita | KPI Lucro | KPI Margem | KPI Ticket médio
GRÁFICO PRINCIPAL: Evolução mensal da receita e do lucro
Receita por categoria        | Margem de lucro por categoria
Receita por estado           | Receita por canal de venda
Ranking de vendedores por lucro
```
Receita e Lucro ficam maiores por serem os KPIs mais importantes; a linha do tempo vem logo abaixo (tendência), depois o diagnóstico (categoria, estado, canal) e por último o detalhamento (vendedores).

## Criação automática do dashboard
A célula abaixo usa o SDK do Databricks para **criar e publicar o dashboard** com todos os KPIs, gráficos e filtros descritos acima. Ela usa uma única base (`vendas`) para que os 4 filtros funcionem em todos os gráficos ao mesmo tempo.

Ao final, aparece o link do dashboard. Ele também fica em **Dashboards** no menu lateral. Se rodar de novo, o dashboard é atualizado.

In [0]:
import json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

TABELA = "workspace.default.vendas"
NOME_DASHBOARD = "Dashboard de Desempenho Comercial - Vendas"
DS = "vendas"

# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def campo(nome, expressao):
    return {
        "name": nome,
        "expression": expressao
    }


def consulta(campos, nome="main_query"):
    return [{
        "name": nome,
        "query": {
            "datasetName": DS,
            "fields": campos,
            "disaggregated": False
        }
    }]


def moldura(titulo, descricao=""):
    m = {
        "showTitle": True,
        "title": titulo
    }

    if descricao:
        m.update({
            "showDescription": True,
            "description": descricao
        })

    return m


def posicao(x, y, largura, altura):
    return {
        "x": x,
        "y": y,
        "width": largura,
        "height": altura
    }


def texto(nome, conteudo, p):
    return {
        "widget": {
            "name": nome,
            "multilineTextboxSpec": {
                "lines": conteudo
            }
        },
        "position": p
    }


def kpi(nome, titulo, c, p, formato=None):
    valor = {
        "fieldName": c["name"],
        "displayName": titulo
    }

    if formato:
        valor["format"] = formato

    return {
        "widget": {
            "name": nome,
            "queries": consulta([c]),
            "spec": {
                "version": 2,
                "widgetType": "counter",
                "encodings": {
                    "value": valor
                },
                "frame": moldura(titulo)
            }
        },
        "position": p
    }


def filtro(nome, tipo, coluna, titulo, p):
    f = coluna.replace(" ", "_")
    q = "filtro_" + f

    return {
        "widget": {
            "name": nome,
            "queries": consulta(
                [campo(f, f"`{coluna}`")],
                q
            ),
            "spec": {
                "version": 2,
                "widgetType": tipo,
                "encodings": {
                    "fields": [{
                        "fieldName": f,
                        "displayName": titulo,
                        "queryName": q
                    }]
                },
                "frame": moldura(titulo)
            }
        },
        "position": p
    }


def barra(nome, titulo, descricao, dimensao, medida, rotulo, p, horizontal=False):
    f = dimensao.replace(" ", "_")

    cat = {
        "fieldName": f,
        "displayName": dimensao
    }

    val = {
        "fieldName": medida["name"],
        "scale": {
            "type": "quantitative"
        },
        "displayName": rotulo
    }

    if horizontal:

        cat["scale"] = {
            "type": "categorical",
            "sort": {
                "by": "x-reversed"
            }
        }

        enc = {
            "x": val,
            "y": cat
        }

    else:

        cat["scale"] = {
            "type": "categorical",
            "sort": {
                "by": "y-reversed"
            }
        }

        enc = {
            "x": cat,
            "y": val
        }

    return {
        "widget": {
            "name": nome,
            "queries": consulta([
                campo(f, f"`{dimensao}`"),
                medida
            ]),
            "spec": {
                "version": 3,
                "widgetType": "bar",
                "encodings": enc,
                "frame": moldura(
                    titulo,
                    descricao
                )
            }
        },
        "position": p
    }

# ============================================================
# MÉTRICAS
# ============================================================

RECEITA = campo(
    "receita",
    "SUM(`Receita`)"
)

LUCRO = campo(
    "lucro",
    "SUM(`Lucro`)"
)

MARGEM = campo(
    "margem",
    "SUM(`Lucro`) * 100 / SUM(`Receita`)"
)

TICKET = campo(
    "ticket",
    "AVG(`Receita`)"
)

MES = campo(
    "mes",
    "DATE_TRUNC('MONTH', `Data`)"
)

# ============================================================
# FORMATOS
# ============================================================

REAIS = {
    "type": "number-currency",
    "currencyCode": "BRL",
    "abbreviation": "compact",
    "decimalPlaces": {
        "type": "max",
        "places": 2
    }
}

PERCENT = {
    "type": "number",
    "decimalPlaces": {
        "type": "fixed",
        "places": 1
    }
}


# ============================================================
# LAYOUT
# ============================================================

layout = [

    # ========================================================
    # CABEÇALHO
    # ========================================================

    texto(
        "titulo",
        """Desempenho Comercial\n
        Jan/2025 → Jul/2026 - Receita, lucro, margem e desempenho comercial\n
        Aluno: Bruno Felipe de Souza Araujo""",
        posicao(
            0, 0, 12, 3
        )
    ),

    # ========================================================
    # FILTROS
    # ========================================================

    filtro(
        "f_periodo",
        "filter-date-range-picker",
        "Data",
        "Período",
        posicao(0, 3, 5, 1)
    ),

    filtro(
        "f_estado",
        "filter-multi-select",
        "Estado",
        "Estado",
        posicao(5, 3, 2, 1)
    ),

    filtro(
        "f_categoria",
        "filter-multi-select",
        "Categoria",
        "Categoria",
        posicao(7, 3, 2, 1)
    ),

    filtro(
        "f_canal",
        "filter-multi-select",
        "Canal de Venda",
        "Canal",
        posicao(9, 3, 3, 1)
    ),

    # ========================================================
    # KPIs
    # ========================================================

    kpi(
        "kpi_receita",
        "Receita total",
        RECEITA,
        posicao(0, 4, 3, 4),
        REAIS
    ),

    kpi(
        "kpi_lucro",
        "Lucro total",
        LUCRO,
        posicao(3, 4, 3, 4),
        REAIS
    ),

    kpi(
        "kpi_margem",
        "Margem (%)",
        MARGEM,
        posicao(6, 4, 3, 4),
        PERCENT
    ),

    kpi(
        "kpi_ticket",
        "Ticket médio",
        TICKET,
        posicao(9, 4, 3, 4),
        REAIS
    ),

    # ========================================================
    # EVOLUÇÃO MENSAL
    # ========================================================

    {
        "widget": {
            "name": "linha_mensal",
            "queries": consulta([
                MES,
                RECEITA,
                LUCRO
            ]),
            "spec": {
                "version": 3,
                "widgetType": "line",
                "encodings": {
                    "x": {
                        "fieldName": "mes",
                        "scale": {
                            "type": "temporal"
                        },
                        "displayName": "Mês"
                    },
                    "y": {
                        "scale": {
                            "type": "quantitative"
                        },
                        "fields": [
                            {
                                "fieldName": "receita",
                                "displayName": "Receita"
                            },
                            {
                                "fieldName": "lucro",
                                "displayName": "Lucro"
                            }
                        ]
                    }
                },
                "frame": moldura(
                    "Evolução mensal",
                    "Receita e lucro ao longo do período selecionado."
                )
            }
        },
        "position": posicao(
            0, 8, 12, 8
        )
    },

    # ========================================================
    # ANÁLISE POR CATEGORIA
    # ========================================================

    barra(
        "bar_categoria",
        "Receita por categoria",
        "Distribuição da receita entre as categorias.",
        "Categoria",
        RECEITA,
        "Receita",
        posicao(0, 16, 6, 8),
        horizontal=True
    ),


    barra(
        "bar_margem_cat",
        "Margem de lucro por categoria",
        "Rentabilidade relativa de cada categoria.",
        "Categoria",
        MARGEM,
        "Margem (%)",
        posicao(6, 16, 6, 8)
    ),

    # ========================================================
    # ANÁLISE GEOGRÁFICA / CANAIS
    # ========================================================

    barra(
        "bar_estado",
        "Receita por estado",
        "Distribuição geográfica da receita.",
        "Estado",
        RECEITA,
        "Receita",
        posicao(0, 24, 6, 8)
    ),
    barra(
        "bar_canal",
        "Receita por canal de venda",
        "Distribuição da receita entre os canais.",
        "Canal de Venda",
        RECEITA,
        "Receita",
        posicao(6, 24, 6, 8)
    ),

    # ========================================================
    # RANKING DE VENDEDORES
    # ========================================================

    barra(
        "bar_vendedor",
        "Lucro por vendedor",
        "Comparação do lucro gerado por vendedor.",
        "Vendedor",
        LUCRO,
        "Lucro",
        posicao(0, 32, 12, 9),
        horizontal=True
    )
]


# ============================================================
# DASHBOARD
# ============================================================

dashboard = {
    "datasets": [
        {
            "name": DS,
            "displayName": "Base de vendas",
            "queryLines": [
                f"SELECT * FROM {TABELA}"
            ]
        }
    ],
    "name": NOME_DASHBOARD,
    "version": 1,
    "isolationMode": "ISOLATION_MODE_SHARED",
    "format": "DASHBOARD_FORMAT_SINGLE_PAGE",
    "allowPublicPreview": True,
    "enablePublicUrl": True,
    "enablePublicLink": True,
    "pages": [
        {
            "name": "visao_geral",
            "displayName": "Visão Geral",
            "pageType": "PAGE_TYPE_CANVAS",
            "layout": layout
        }
    ]
}

# ============================================================
# WAREHOUSE
# ============================================================

warehouse_id = next(
    iter(w.warehouses.list())
).id

# ============================================================
# CAMINHO
# ============================================================

pasta = (
    f"/Users/{w.current_user.me().user_name}"
)

# ============================================================
# CORPO DA REQUISIÇÃO
# ============================================================

corpo = {
    "display_name": NOME_DASHBOARD,
    "warehouse_id": warehouse_id,
    "serialized_dashboard": json.dumps(
        dashboard,
        ensure_ascii=False
    )
}

# ============================================================
# LOCALIZA DASHBOARD EXISTENTE
# ============================================================

existentes = w.api_client.do("GET", "/api/2.0/lakeview/dashboards", query={"page_size": 100}).get("dashboards", [])
existente = next(
    (
        d
        for d in existentes
        if d.get("display_name") == NOME_DASHBOARD
    ),
    None
)

# ============================================================
# CRIA OU ATUALIZA
# ============================================================

if existente:

    dashboard_id = existente[
        "dashboard_id"
    ]
    w.api_client.do(
        "PATCH",
        f"/api/2.0/lakeview/dashboards/{dashboard_id}",
        body=corpo
    )
    print(
        "Dashboard atualizado."
    )
else:
    criado = w.api_client.do(
        "POST",
        "/api/2.0/lakeview/dashboards",
        body={
            **corpo,
            "parent_path": pasta
        }
    )
    dashboard_id = criado[
        "dashboard_id"
    ]
    print(
        "Dashboard criado."
    )

# ============================================================
# PUBLICAÇÃO
# ============================================================

w.api_client.do(
    "POST",
    f"/api/2.0/lakeview/dashboards/{dashboard_id}/published",
    body={
        "warehouse_id": warehouse_id,
        "embed_credentials": True
    }
)

# ============================================================
# LINK
# ============================================================

url = (
    f"{w.config.host.rstrip('/')}"
    f"/dashboardsv3/{dashboard_id}/published"
)

displayHTML(f'''
    <div style="
        margin-top: 15px;
        padding: 12px 16px;
        border-radius: 8px;
        background-color: #f5f5f5;
        border: 1px solid #ddd;
    ">
        <a href="{url}"
            target="_blank"
            style="
                font-weight: 600;
                text-decoration: none;">
            {NOME_DASHBOARD}
        </a>
    </div>
    '''
)

Dashboard atualizado.


Dashboard de Desempenho Comercial - Vendas

## Parte 5 – Interatividade
| Filtro | Por que ajuda na análise |
|---|---|
| **Período** (Data) | Permite comparar meses/anos e investigar quedas pontuais (ex.: mar/2025, pior mês). |
| **Estado** | Permite ao gestor regional ver só a sua região e comparar com o total. |
| **Categoria** | Permite ver tendência, estados e vendedores de uma categoria específica. |
| **Canal de venda** | Permite avaliar se um canal tem comportamento diferente dos demais. |

Além dos filtros, ao **clicar** em uma barra ou ponto o Databricks aplica *cross-filter* nos outros gráficos.

**Teste de combinação (Consulta 6 – SP + Site):** a receita cai para R$ 5,85 mi, a margem fica em 29,2% e o padrão se mantém (Informática 47,8% da receita com margem de 27,7%; Acessórios com margem de 48,2%). Os resultados continuam coerentes com a visão geral.

## Parte 8 – Teste do dashboard (visão do gestor)
1. **Desempenho geral:** R$ 38,0 mi de receita, R$ 11,1 mi de lucro, margem de 29,1% e ticket médio de R$ 3,8 mil (10.000 vendas).
2. **Melhores categorias:** Informática lidera em receita (R$ 17,5 mi); Acessórios tem a melhor margem (48,1%).
3. **Regiões que merecem atenção:** SP concentra 49,5% da receita; MG é o estado com menor receita (11,8%). As margens são parecidas (~29%).
4. **Tendência:** crescimento: jan-jul/2026 teve +8,1% de receita e +7,9% de lucro em relação a jan-jul/2025. Pior mês: mar/2025; melhor: ago/2025.
5. **Destaques:** Site é o maior canal (29,8%); Bruno Costa lidera em lucro (R$ 990 mil), mas a equipe é equilibrada.
6. **Receita alta = lucro alto?** Não: Informática tem a maior receita e margem de 27,9%, enquanto Acessórios tem a menor receita e a maior margem (48,1%). Componentes tem a pior margem (24,1%).

## 3 Insights encontrados
1. **Receita alta não significa margem alta:** Informática gera 45,9% da receita com margem de 27,9%; Acessórios gera só 4,1% da receita, mas com margem de 48,1%. *Decisão:* incentivar a venda de acessórios junto com notebooks/smartphones e revisar custo/preço de Componentes (margem de 24,1%).
2. **Dependência de São Paulo:** SP sozinho representa 49,5% da receita, enquanto cada outro estado fica entre 11,8% e 13,3%, todos com margem de ~29%. *Decisão:* como a rentabilidade é igual, há espaço para expandir vendas em MG, SC, RJ e PR e reduzir o risco de concentração.
3. **Crescimento saudável:** de jan a jul, a receita passou de R$ 13,42 mi (2025) para R$ 14,51 mi (2026), +8,1%, e o lucro cresceu 7,9%, com margem estável. *Decisão:* manter a estratégia atual e usar o canal Site (maior canal) para acelerar o crescimento; Televendas (5,3%) deve ser reavaliado.